
# 📘 Day 6 学习笔记：MoE 混合专家模型 —— Router 如何分发 Token

> 学习主线：**Dense FFN → Sparse MoE → Router / Gate → Top-K Expert → Load Balancing → Shared Expert → Expert Parallelism → Tool Router 工程映射**

---

## 🎯 今日学习大纲

### 1. 你今天要解决的核心问题

今天不只是“知道 MoE 是混合专家”，而是要真正能解释：

- MoE 为什么通常替换的是 **FFN / MLP 层**，不是 Attention 层？
- 为什么 MoE 可以做到 **总参数量很大，但每 token 计算量不一定大**？
- Router / Gate 是怎么把 token 分配给 expert 的？
- Top-1 / Top-2 routing 的区别是什么？
- 为什么会出现专家负载不均衡？
- 为什么需要 load balancing loss？
- 为什么 DeepSeekMoE 要引入 shared experts？
- 为什么 MoE 推理时可能显存很大，但速度不一定快？
- MoE 和 LangGraph / Agent Router / Tool Router 有什么工程类比？

---

## ✅ 今日完成目标

学完这份 Notebook，你应该能做到：

1. 画出 Dense FFN 和 MoE FFN 的结构差异。
2. 写出 Router 的基本公式：

\[
\text{router\_logits} = h W_r
\]

\[
p = \text{softmax}(\text{router\_logits})
\]

\[
\text{TopKExperts} = \operatorname{topk}(p, k)
\]

3. 用 NumPy 写一个 toy router，把 token 分配给 top-2 experts。
4. 解释为什么 MoE 的 **total parameters** 和 **active parameters** 不一样。
5. 通过可视化观察专家负载不均衡。
6. 理解 capacity factor、token dropping、load balancing 的动机。
7. 解释 shared expert 为什么能承载通用知识。
8. 把 MoE Router 映射到 Tool Router / Agent Router。

---

## 🧠 今日核心概念

| 概念 | 你需要掌握的解释 |
|---|---|
| Dense FFN | 每个 token 都经过同一个 FFN |
| Sparse MoE | 每个 token 只激活少数几个 expert FFN |
| Router / Gate | 根据 token hidden state 选择专家 |
| Top-K Routing | 从 N 个 expert 中选择 K 个参与计算 |
| Active Parameters | 当前 token 实际参与计算的参数 |
| Total Parameters | 模型所有专家参数的总量 |
| Load Balancing | 避免所有 token 都挤向少数专家 |
| Capacity Factor | 每个 expert 每批最多接收多少 token |
| Shared Expert | 总是激活的通用专家，承载共性知识 |
| Expert Parallelism | 把不同专家分布到不同设备上并行处理 |


In [ ]:

# Day 6 所需基础库
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print("环境就绪 ✅")



---

# 1️⃣ 从 Dense FFN 开始：MoE 到底替换了 Transformer 的哪一块？

一个标准 Transformer block 可以粗略写成：

```text
x
 ↓
Self-Attention
 ↓
FFN / MLP
 ↓
output
```

在大多数 LLM 的 MoE 架构里，MoE 主要替换的是：

```text
FFN / MLP
```

而不是 Attention。

---

## 1.1 Dense FFN 的形式

标准 Dense FFN 通常是：

\[
\text{FFN}(x) = W_2 \sigma(W_1 x)
\]

其中：

- \(x \in \mathbb{R}^{d_{model}}\)
- \(W_1 \in \mathbb{R}^{d_{ff} \times d_{model}}\)
- \(W_2 \in \mathbb{R}^{d_{model} \times d_{ff}}\)
- \(\sigma\) 是激活函数，例如 GELU / SiLU

对每个 token 来说：

```text
所有 token 都经过同一个 FFN
```

这叫 **dense computation**：所有 FFN 参数都会被激活。

---

## 1.2 MoE FFN 的形式

MoE 把一个 FFN 替换成多个 expert FFN：

\[
E_1(x), E_2(x), ..., E_N(x)
\]

每个 \(E_i\) 都可以是一个 FFN：

\[
E_i(x) = W_{2,i} \sigma(W_{1,i}x)
\]

但关键是：每个 token 不会跑全部 experts，而是由 router 选择少数几个：

\[
\text{MoE}(x) = \sum_{i \in \operatorname{TopK}(p)} p_i E_i(x)
\]

其中：

\[
p = \text{softmax}(xW_r)
\]

---

## 1.3 一句话直觉

Dense FFN 是：

```text
所有 token 都去同一个办公室办事
```

MoE 是：

```text
每个 token 先去问前台 Router：我该找哪几个专家？
然后只去被选中的专家那里处理。
```

所以 MoE 的核心不是“多个模型投票”，而是：

> **在 Transformer 内部，用 Router 对 token 做条件计算，只激活部分专家 FFN。**


In [ ]:

pd.DataFrame([
    ["Dense FFN", "1 个 FFN", "全部 token 都走同一套 FFN", "所有 FFN 参数", "实现简单，计算稳定"],
    ["Sparse MoE", "N 个 expert FFN", "每个 token 只走 Top-K expert", "少数 expert 参数", "总容量大，但计算可控"],
], columns=["结构", "FFN 数量", "Token 路径", "每 token 激活参数", "核心特征"])



---

# 2️⃣ MoE 的核心公式：Router / Gate 如何选专家？

设某一层某个 token 的 hidden state 是：

\[
h \in \mathbb{R}^{d_{model}}
\]

Router 是一个线性层：

\[
W_r \in \mathbb{R}^{d_{model} \times N}
\]

其中 \(N\) 是 expert 数量。

---

## 2.1 Router logits

\[
z = h W_r
\]

\[
z \in \mathbb{R}^{N}
\]

它表示这个 token 对每个 expert 的原始打分。

---

## 2.2 Router probability

\[
p_i = \frac{e^{z_i}}{\sum_{j=1}^{N}e^{z_j}}
\]

即：

\[
p = \text{softmax}(z)
\]

---

## 2.3 Top-K Routing

选择概率最高的 K 个 experts：

\[
S = \operatorname{TopK}(p, K)
\]

最后输出：

\[
y = \sum_{i \in S} \alpha_i E_i(h)
\]

其中 \(\alpha_i\) 是归一化后的 expert 权重。

---

## 2.4 Top-1 vs Top-2

### Top-1 Routing

\[
y = E_{i^*}(h), \quad i^* = \arg\max_i p_i
\]

优点：
- 计算最省
- routing 简单

缺点：
- 表达能力可能弱
- expert 选择更硬

### Top-2 Routing

\[
y = \alpha_a E_a(h) + \alpha_b E_b(h)
\]

优点：
- 表达更平滑
- 两个专家可以互补

缺点：
- 每 token FFN 计算约增加到 2 个专家
- 调度和通信更复杂

---

## 2.5 面试一句话

> Router 不是在句子级别选专家，而是通常在 **token 级别、layer 级别** 选择专家；同一句话里的不同 token 可能走不同专家，同一个 token 在不同层也可能走不同专家。


In [ ]:

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def topk_router(hidden_states, router_w, k=2):
    # hidden_states: [T, d_model]
    # router_w: [d_model, num_experts]
    logits = hidden_states @ router_w
    probs = softmax(logits, axis=-1)
    topk_idx = np.argsort(probs, axis=-1)[:, -k:][:, ::-1]
    topk_probs = np.take_along_axis(probs, topk_idx, axis=-1)
    topk_probs_norm = topk_probs / topk_probs.sum(axis=-1, keepdims=True)
    return logits, probs, topk_idx, topk_probs_norm

T = 16
d_model = 12
num_experts = 6
top_k = 2

hidden_states = np.random.randn(T, d_model)
router_w = np.random.randn(d_model, num_experts) * 0.8

logits, probs, topk_idx, topk_probs_norm = topk_router(hidden_states, router_w, k=top_k)

routing_df = pd.DataFrame({
    "token_id": np.arange(T),
    "top1_expert": topk_idx[:, 0],
    "top1_weight": topk_probs_norm[:, 0].round(3),
    "top2_expert": topk_idx[:, 1],
    "top2_weight": topk_probs_norm[:, 1].round(3),
})
routing_df


In [ ]:

plt.figure(figsize=(9, 5))
plt.imshow(probs, aspect='auto')
plt.colorbar(label='router probability')
plt.xlabel('Expert ID')
plt.ylabel('Token ID')
plt.title('Router probability heatmap: 每个 token 对各 expert 的选择概率')

for t in range(T):
    for j in topk_idx[t]:
        plt.text(j, t, '★', ha='center', va='center', fontsize=10)

plt.show()



---

# 3️⃣ Active Parameters vs Total Parameters：MoE 最容易被问的点

## 3.1 Dense 模型

假设一个 Dense FFN 参数量为：

\[
P_{ffn}
\]

每个 token 都激活它，所以：

```text
total parameters ≈ active parameters
```

---

## 3.2 MoE 模型

假设有 \(N\) 个 experts，每个 expert 参数量都是 \(P_{ffn}\)。

总 expert 参数量：

\[
P_{total} = N \cdot P_{ffn}
\]

但如果每个 token 只激活 Top-K 个 expert，则每 token 激活参数约为：

\[
P_{active} = K \cdot P_{ffn}
\]

因此：

\[
\frac{P_{active}}{P_{total}} = \frac{K}{N}
\]

---

## 3.3 例子

如果：

- experts = 8
- top_k = 2

那么每个 token 只激活：

\[
\frac{2}{8} = 25\%
\]

的 expert 参数。

这就是为什么 MoE 可以：

```text
总参数量很大
但每 token 计算量不按总参数量线性增长
```

---

## 3.4 但是不要误解

MoE 并不是“显存一定省”。

原因是：

- 所有 expert 权重通常仍要加载到显存或分布在多卡上；
- token 到 expert 的 dispatch / gather 有通信成本；
- expert 负载不均衡会导致某些设备等待；
- batch 太小时，专家并行效率很差。

所以更准确的说法是：

> **MoE 主要省的是每 token 计算量，不一定省总显存；它用路由复杂度和系统调度复杂度换取更大的模型容量。**


In [ ]:

def moe_active_ratio(num_experts, top_k):
    return top_k / num_experts

settings = []
for n in [4, 8, 16, 32, 64]:
    for k in [1, 2, 4]:
        if k <= n:
            settings.append({
                "num_experts": n,
                "top_k": k,
                "active_ratio": moe_active_ratio(n, k),
                "active_percent": f"{100 * moe_active_ratio(n, k):.1f}%"
            })

pd.DataFrame(settings)


In [ ]:

expert_counts = np.array([4, 8, 16, 32, 64])
plt.figure(figsize=(8, 4.8))

for k in [1, 2, 4]:
    vals = [k / n if k <= n else np.nan for n in expert_counts]
    plt.plot(expert_counts, vals, marker='o', label=f'top_k={k}')

plt.xlabel('Number of Experts')
plt.ylabel('Active Expert Ratio')
plt.title('专家越多，在固定 top_k 下每 token 激活比例越低')
plt.grid(alpha=0.3)
plt.legend()
plt.show()



---

# 4️⃣ Toy MoE Forward：真的把 token 送进专家

现在我们写一个最小 MoE layer：

1. Router 计算每个 token 的 expert 概率。
2. 选 Top-K experts。
3. 每个 expert 是一个简单 MLP。
4. 每个 token 的输出是 Top-K expert 输出的加权和。

为了便于学习，这里用 NumPy 实现，不追求训练，只追求理解数据流。

---

## 4.1 Expert FFN 公式

每个 expert：

\[
E_i(h) = W_{2,i} \cdot \text{ReLU}(W_{1,i}h)
\]

最后输出：

\[
y_t = \sum_{i \in S_t} \alpha_{t,i} E_i(h_t)
\]

其中：

- \(t\) 是 token index
- \(S_t\) 是 token \(t\) 选中的 top-k experts
- \(\alpha_{t,i}\) 是归一化 gating weight


In [ ]:

def relu(x):
    return np.maximum(0, x)

class ToyExpert:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) / math.sqrt(d_model)
        self.W2 = np.random.randn(d_ff, d_model) / math.sqrt(d_ff)

    def __call__(self, x):
        # x: [d_model]
        return relu(x @ self.W1) @ self.W2

class ToyMoELayer:
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        self.d_model = d_model
        self.num_experts = num_experts
        self.top_k = top_k
        self.router_w = np.random.randn(d_model, num_experts) / math.sqrt(d_model)
        self.experts = [ToyExpert(d_model, d_ff) for _ in range(num_experts)]

    def forward(self, hidden_states):
        logits, probs, topk_idx, topk_weights = topk_router(hidden_states, self.router_w, self.top_k)
        outputs = []
        expert_calls = {i: 0 for i in range(self.num_experts)}

        for t, h in enumerate(hidden_states):
            y = np.zeros_like(h)
            for slot in range(self.top_k):
                expert_id = int(topk_idx[t, slot])
                weight = topk_weights[t, slot]
                y += weight * self.experts[expert_id](h)
                expert_calls[expert_id] += 1
            outputs.append(y)

        return np.stack(outputs), probs, topk_idx, topk_weights, expert_calls

moe = ToyMoELayer(d_model=12, d_ff=32, num_experts=6, top_k=2)
out, probs2, topk_idx2, topk_weights2, expert_calls = moe.forward(hidden_states)

print("输入形状:", hidden_states.shape)
print("输出形状:", out.shape)
print("每个 expert 被调用次数:", expert_calls)


In [ ]:

expert_ids = list(expert_calls.keys())
loads = list(expert_calls.values())

plt.figure(figsize=(8, 4.5))
plt.bar(expert_ids, loads)
for x, y in zip(expert_ids, loads):
    plt.text(x, y + 0.1, str(y), ha='center')
plt.xlabel('Expert ID')
plt.ylabel('Number of token assignments')
plt.title('Toy MoE: 每个 expert 被分配到的 token 次数')
plt.show()



---

# 5️⃣ MoE 的训练难点：专家负载不均衡

MoE 最核心的训练难点之一是：

```text
Router 可能总是把 token 分给少数几个专家。
```

这会导致：

- 热门 expert 过载；
- 冷门 expert 学不到东西；
- 多卡 expert parallelism 下，某些 GPU 忙死，某些 GPU 空闲；
- 系统吞吐被最慢的专家拖住。

---

## 5.1 为什么会不均衡？

因为 router 是可学习的。

在训练早期，如果某个 expert 偶然更容易降低 loss，router 可能越来越偏向它：

```text
更多 token → 该 expert 更新更多 → 它更强 → router 更爱选它 → 更加过载
```

这会形成“富者愈富”的反馈。

---

## 5.2 Load Balancing Loss 的直觉

理想情况是：

```text
所有 expert 都有相对均匀的 token 分配
```

所以训练时通常会额外加一个辅助 loss，鼓励 router 不要塌缩到少数专家。

非常简化地说，可以关注两个量：

- \(f_i\)：实际被分给 expert \(i\) 的 token 比例；
- \(p_i\)：router 对 expert \(i\) 的平均概率。

希望它们都不要过度偏向少数专家。

一种常见直觉形式是：

\[
L_{balance} \propto N \sum_{i=1}^{N} f_i p_i
\]

这里不要求你背具体论文里的每个实现细节，但必须能说出动机：

> **load balancing loss 用来防止 router 把大量 token 塞给少数 expert，从而让专家训练更充分，也让系统吞吐更稳定。**


In [ ]:

T_big = 256
d_model_big = 16
num_experts_big = 8
top_k_big = 2

hidden_big = np.random.randn(T_big, d_model_big)

# 人为制造偏置：让 expert 0 和 1 的 router weight 更大
router_w_biased = np.random.randn(d_model_big, num_experts_big) * 0.2
router_w_biased[:, 0] += 1.2
router_w_biased[:, 1] += 0.8

_, probs_biased, topk_biased, _ = topk_router(hidden_big, router_w_biased, k=top_k_big)

loads = np.zeros(num_experts_big, dtype=int)
for row in topk_biased:
    for e in row:
        loads[e] += 1

plt.figure(figsize=(8, 4.5))
plt.bar(np.arange(num_experts_big), loads)
for x, y in enumerate(loads):
    plt.text(x, y + 1, str(y), ha='center')
plt.xlabel('Expert ID')
plt.ylabel('Assignments')
plt.title('Router 偏置导致专家负载不均衡')
plt.show()

print("负载标准差：", loads.std().round(2))
print("最大负载 / 平均负载：", (loads.max() / loads.mean()).round(2))


In [ ]:

def simple_load_balance_metric(probs, topk_idx, num_experts):
    f = np.zeros(num_experts)
    for row in topk_idx:
        for e in row:
            f[int(e)] += 1
    f = f / f.sum()
    p = probs.mean(axis=0)
    metric = num_experts * np.sum(f * p)
    return metric, f, p

metric, f, p = simple_load_balance_metric(probs_biased, topk_biased, num_experts_big)

balance_df = pd.DataFrame({
    "expert": np.arange(num_experts_big),
    "actual_assignment_fraction_f": f.round(3),
    "mean_router_probability_p": p.round(3),
})
print("简化 load balance metric:", round(metric, 4))
balance_df



---

# 6️⃣ Capacity Factor 与 Token Dropping

在实际系统里，每个 expert 一般不能无限接收 token。

否则如果 expert 0 被分到 80% token，它会变成系统瓶颈。

因此 MoE 里常有一个概念：

```text
expert capacity
```

简单理解：

> 每个 expert 在一个 batch 中最多能处理多少 token。

---

## 6.1 Capacity 的简化计算

设：

- tokens 数量为 \(T\)
- experts 数量为 \(N\)
- top-k 为 \(K\)
- capacity factor 为 \(c\)

平均每个 expert 应该处理：

\[
\frac{T \cdot K}{N}
\]

设置 capacity：

\[
\text{capacity} = \left\lceil c \cdot \frac{T \cdot K}{N} \right\rceil
\]

如果某 expert 分到的 token 超过 capacity，多出来的 token 可能会：

- 被 drop；
- 被 reroute；
- 使用 backup expert；
- 或在不同实现中采用其他策略。

---

## 6.2 为什么这很重要？

因为 MoE 不是单纯算法问题，也是系统调度问题。

```text
Router 选择专家
   ↓
tokens dispatch 到 expert
   ↓
expert 并行计算
   ↓
outputs gather 回原顺序
```

如果某个 expert 负载过重：

```text
其他专家算完等它
```

整个 MoE layer 的延迟就会被它拖住。


In [ ]:

def simulate_capacity(topk_idx, num_experts, capacity_factor=1.0):
    T, K = topk_idx.shape
    capacity = math.ceil(capacity_factor * (T * K / num_experts))
    expert_load = {i: 0 for i in range(num_experts)}
    kept = 0
    dropped = 0
    dropped_records = []

    for t in range(T):
        for slot in range(K):
            e = int(topk_idx[t, slot])
            if expert_load[e] < capacity:
                expert_load[e] += 1
                kept += 1
            else:
                dropped += 1
                dropped_records.append((t, e))

    return capacity, expert_load, kept, dropped, dropped_records

for cf in [0.75, 1.0, 1.25, 1.5, 2.0]:
    capacity, expert_load, kept, dropped, _ = simulate_capacity(topk_biased, num_experts_big, capacity_factor=cf)
    print(f"capacity_factor={cf:.2f} | capacity={capacity} | kept={kept} | dropped={dropped}")


In [ ]:

cfs = np.array([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0])
drops = []
for cf in cfs:
    capacity, expert_load, kept, dropped, _ = simulate_capacity(topk_biased, num_experts_big, capacity_factor=cf)
    drops.append(dropped)

plt.figure(figsize=(8, 4.5))
plt.plot(cfs, drops, marker='o')
for x, y in zip(cfs, drops):
    plt.text(x, y + 2, str(y), ha='center')
plt.xlabel('Capacity Factor')
plt.ylabel('Dropped Assignments')
plt.title('Capacity Factor 越小，过载 expert 的 token 越容易被 drop')
plt.grid(alpha=0.3)
plt.show()



---

# 7️⃣ Shared Expert：为什么 DeepSeekMoE 要引入共享专家？

传统 MoE 的问题之一是：

```text
所有知识都被迫分散到 routed experts 里。
```

但语言模型里有很多“所有 token 都可能需要”的通用能力，例如：

- 基础语法
- 常识
- 通用推理
- 常见模式
- 通用格式处理

如果这些共性知识也分散到 routed experts 里，就可能造成冗余：

```text
每个专家都学一份通用知识
```

---

## 7.1 Shared Expert 的直觉

Shared expert 是：

```text
不经过 router 选择，总是参与处理的专家。
```

输出可以写成：

\[
y = E_{shared}(h) + \sum_{i \in \operatorname{TopK}(p)} \alpha_i E_i(h)
\]

也就是：

```text
通用专家负责通用能力
路由专家负责更细的专门能力
```

---

## 7.2 类比

没有 shared expert：

```text
每个科室都要自己培养一个前台、行政、基础接待能力。
```

有 shared expert：

```text
医院有统一前台和公共基础服务，
各科室专注处理专科问题。
```

---

## 7.3 面试回答要点

如果被问：

> shared expert 有什么用？

你可以答：

- 它用于捕获通用知识；
- 避免 routed experts 重复学习大量共性模式；
- 让 routed experts 更专注于差异化、细粒度能力；
- 有助于提升 expert specialization。


In [ ]:

class ToySharedMoELayer(ToyMoELayer):
    def __init__(self, d_model, d_ff, num_experts, top_k=2, shared_weight=1.0):
        super().__init__(d_model, d_ff, num_experts, top_k)
        self.shared_expert = ToyExpert(d_model, d_ff)
        self.shared_weight = shared_weight

    def forward(self, hidden_states):
        logits, probs, topk_idx, topk_weights = topk_router(hidden_states, self.router_w, self.top_k)
        outputs = []
        for t, h in enumerate(hidden_states):
            y = self.shared_weight * self.shared_expert(h)
            for slot in range(self.top_k):
                expert_id = int(topk_idx[t, slot])
                weight = topk_weights[t, slot]
                y += weight * self.experts[expert_id](h)
            outputs.append(y)
        return np.stack(outputs), probs, topk_idx, topk_weights

shared_moe = ToySharedMoELayer(d_model=12, d_ff=32, num_experts=6, top_k=2, shared_weight=0.5)
shared_out, shared_probs, shared_topk, shared_weights = shared_moe.forward(hidden_states)

print("普通 MoE 输出形状:", out.shape)
print("Shared MoE 输出形状:", shared_out.shape)
print("两者输出均值差异:", np.mean(np.abs(out - shared_out)).round(4))



---

# 8️⃣ Expert Parallelism：MoE 为什么对系统实现要求高？

MoE 的计算图不是简单的：

```text
所有 token 经过同一层
```

而是：

```text
token 先被 router 分发到不同 expert
expert 可能位于不同 GPU
计算完成后再 gather 回原来的 token 顺序
```

---

## 8.1 系统链路

```text
hidden states
   ↓
router logits
   ↓
top-k expert selection
   ↓
dispatch tokens to experts
   ↓
expert FFN computation
   ↓
combine / gather outputs
   ↓
next transformer layer
```

---

## 8.2 为什么 MoE 推理时显存大？

因为虽然每个 token 只激活少数 experts，但系统通常仍要让 experts 的权重可用：

- 单机：多个 expert 权重都在显存里；
- 多机多卡：experts 分布在不同 GPU 上；
- 请求进来后，token 可能被路由到任意 expert。

所以：

> **MoE 节省 active compute，但不等于节省 total weights memory。**

---

## 8.3 为什么 MoE 速度不一定快？

常见原因：

1. token dispatch / gather 有额外开销；
2. expert 负载不均衡；
3. batch 太小，expert 并行效率低；
4. 跨卡通信成本高；
5. 服务框架 kernel 和并行策略不够成熟。

---

## 8.4 排障思路

如果 MoE 模型推理慢，你不能只看总参数量或 active 参数量，要看：

```text
1. top_k 是多少
2. expert 数量是多少
3. expert 是否跨卡
4. router 是否负载均衡
5. batch size 是否足够大
6. dispatch/gather 是否成为瓶颈
7. 是否有 expert parallelism 优化
```



---

# 9️⃣ 应用工程映射：MoE Router vs Agent Tool Router

你这 7 天路线要求每天都做工程映射。Day 6 对应的是：

```text
Tool Router / Agent Router / 多工具选择与失败回退
```

---

## 9.1 MoE Router 的模式

```text
token hidden state
   ↓
router
   ↓
选择 top-k experts
   ↓
专家处理
   ↓
结果合并
```

---

## 9.2 Agent Tool Router 的模式

```text
user request / state
   ↓
router / planner
   ↓
选择 tool(s)
   ↓
工具执行
   ↓
结果合并或进入下一轮
```

---

## 9.3 类比关系

| MoE | Agent / LangGraph |
|---|---|
| Token | User request / intermediate state |
| Expert | Tool / sub-agent / skill |
| Router | Planner / classifier / routing node |
| Top-K expert | Top-K tools |
| Load balancing | 工具限流 / 并发控制 / fallback |
| Expert overload | 某个工具超时、失败或队列阻塞 |
| Shared expert | 通用回答器 / common reasoning node |

---

## 9.4 但不要过度类比

MoE Router 是：

```text
模型内部的、token 级别的、可微分/可训练的路由
```

Agent Tool Router 是：

```text
应用层的、请求级别或步骤级别的、通常由规则/LLM/分类器控制的路由
```

它们不是同一个东西，但架构直觉很像：

> **都在做“按输入选择合适处理单元”，并且都要处理负载、失败、回退和可观测性。**


In [ ]:

tools = {
    "retriever": "检索知识库",
    "calculator": "做确定性计算",
    "web_search": "查最新信息",
    "code_executor": "运行代码",
    "human_review": "请求人工复核",
}

queries = [
    "帮我计算 KV Cache 显存",
    "查一下最新的 Mixtral 架构说明",
    "运行这个 Python demo",
    "这段论文结论是否合理",
    "从我的文档里找一下 MoE 计划",
]

def simple_tool_router(query):
    if "计算" in query or "显存" in query:
        return ["calculator"]
    if "最新" in query or "查一下" in query:
        return ["web_search"]
    if "运行" in query or "Python" in query:
        return ["code_executor"]
    if "文档" in query or "找一下" in query:
        return ["retriever"]
    if "是否合理" in query or "复核" in query:
        return ["human_review"]
    return ["retriever"]

pd.DataFrame([
    {"query": q, "selected_tools": ", ".join(simple_tool_router(q))}
    for q in queries
])



---

# 🔟 今日面试高频问答

## Q1：MoE 和 Dense Transformer 的核心区别是什么？

**满分回答：**

Dense Transformer 的每个 token 都经过同一个 FFN；MoE 把 FFN 替换成多个 expert FFN，并用 router 为每个 token 选择 top-k experts。这样模型总参数量可以很大，但每个 token 只激活少数专家，所以 active compute 可控。

---

## Q2：MoE 为什么参数量大，但计算量不一定大？

**满分回答：**

MoE 的 total parameters 是所有 expert 参数总和，但每个 token 只经过 top-k experts，因此每 token active parameters 远小于 total parameters。计算量主要取决于 active experts，而不是所有 experts。但是权重显存、通信和调度开销仍然存在，所以 MoE 不一定省显存。

---

## Q3：Router 的输入输出是什么？

**满分回答：**

Router 的输入通常是 token 的 hidden state \(h\)，经过一个线性层得到每个 expert 的 logits：

\[
z = hW_r
\]

再通过 softmax 得到 expert probability，最后选择 top-k experts 参与该 token 的 FFN 计算。

---

## Q4：为什么 MoE 会有 load balancing 问题？

**满分回答：**

Router 可能倾向于把大部分 token 分给少数几个 expert，导致热门 expert 过载，冷门 expert 训练不足。系统上也会造成某些 GPU 忙、某些 GPU 空闲。因此训练时通常引入 load balancing loss 或其他路由约束。

---

## Q5：Capacity Factor 是什么？

**满分回答：**

Capacity factor 用来限制每个 expert 在一个 batch 中最多接收多少 token。它通常基于平均 token 分配数乘以一个系数。如果某个 expert 超出容量，多余 token 可能被 drop 或 reroute。它是控制 expert 负载和系统延迟的重要机制。

---

## Q6：Shared Expert 的作用是什么？

**满分回答：**

Shared expert 总是参与处理，负责承载通用知识和共性模式，从而减少 routed experts 重复学习通用能力，让 routed experts 更专注于细粒度、专门化知识。

---

## Q7：MoE 为什么推理时可能显存大但速度没想象中快？

**满分回答：**

因为虽然每个 token 只激活少数专家，但 expert 权重通常仍要驻留显存或分布在多卡上；同时 token dispatch/gather、跨卡通信、负载不均衡和小 batch 都会拖慢吞吐。

---

## Q8：MoE 和 Agent Tool Router 有什么相似点？

**满分回答：**

两者都根据输入选择合适的处理单元。MoE 根据 token hidden state 选择 experts；Agent 根据用户请求或中间 state 选择 tools/sub-agents。但 MoE 是模型内部可训练路由，Agent Router 是应用层编排路由。



---

# 1️⃣1️⃣ 今日最小作业

## 作业 1：改 top_k

把 toy router 的 `top_k` 从 2 改成：

```text
top_k = 1
top_k = 4
```

观察：

- 每个 expert 的负载如何变化？
- active parameter ratio 如何变化？
- 你会如何解释质量和计算量的 trade-off？

---

## 作业 2：制造 router collapse

修改 `router_w_biased`，让 expert 0 更强：

```python
router_w_biased[:, 0] += 2.0
```

观察：

- expert 0 是否严重过载？
- capacity factor 多大才不会 drop 太多 token？

---

## 作业 3：写一段面试回答

题目：

> 为什么 MoE 模型不是“参数越大推理越慢”？

你需要包含：

- total parameters
- active parameters
- top-k routing
- 显存和通信的 caveat

---

## 作业 4：工程映射

写一个 10 行以内的小结：

> 如果你在 LangGraph 里设计一个 Tool Router，你会如何借鉴 MoE 的思想？你又会如何避免“某个工具被过度调用”？



---

# 1️⃣2️⃣ 推荐学习平台 / 视频 / 仓库（3–5 个高质量资源）

## 资源 1：Mistral AI - Mixtral of Experts
- 类型：官方技术说明
- 用途：理解 Mixtral 这类 Sparse MoE 的真实架构，尤其是每层每 token 选 2 个 experts 的机制。
- 链接：https://mistral.ai/news/mixtral-of-experts

## 资源 2：Hugging Face Blog - Mixture of Experts Explained
- 类型：图文教程
- 用途：适合复习 MoE 的 building blocks、训练难点与 serving tradeoff。
- 链接：https://huggingface.co/blog/moe

## 资源 3：DeepSeekMoE Paper
- 类型：论文
- 用途：理解 fine-grained expert segmentation 与 shared expert isolation。
- 链接：https://arxiv.org/abs/2401.06066

## 资源 4：MoE Token Routing Explained
- 类型：YouTube / ClassCentral 索引视频
- 用途：看 token routing、router logits、top-k selection 的代码直觉。
- 链接：https://www.classcentral.com/course/youtube-moe-token-routing-explained-how-mixture-of-experts-works-with-code-522704

## 资源 5：lucidrains / mixture-of-experts
- 类型：GitHub 代码仓库
- 用途：看 PyTorch 版 Sparsely Gated MoE 实现。
- 链接：https://github.com/lucidrains/mixture-of-experts

---

# ✅ 今日一句话总总结

> **MoE 的本质是在 Transformer FFN 位置引入条件计算：Router 为每个 token 选择少数专家，使模型拥有更大的总容量，同时保持每 token active compute 可控；但它带来了路由、负载均衡、通信、显存和 serving 调度复杂度。**
